In [ ]:
# Install necessary libraries
!pip install -q transformers datasets torch scikit-learn pandas accelerate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline
)
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import torch.nn.functional as F
import os

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Configuration
MODEL_CHECKPOINT = "dmis-lab/biobert-base-cased-v1.1"
SAVE_PATH = "./saved_ade_model"  # Directory to save the model
BATCH_SIZE = 16
MAX_LEN = 128

Using device: cuda


Data Preprocessong

In [ ]:
# Load the ADE Corpus V2 dataset
dataset = load_dataset("ade_corpus_v2", "Ade_corpus_v2_classification")

# Split: 70% Train, 15% Validation, 15% Test
train_testvalid = dataset['train'].train_test_split(test_size=0.3, seed=42)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

datasets = {
    'train': train_testvalid['train'],
    'validation': test_valid['train'],
    'test': test_valid['test']
}

print(f"Train size: {len(datasets['train'])}")
print(f"Validation size: {len(datasets['validation'])}")
print(f"Test size: {len(datasets['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Ade_corpus_v2_classification/train-00000(…):   0%|          | 0.00/1.71M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/23516 [00:00<?, ? examples/s]

Train size: 16461
Validation size: 3527
Test size: 3528


Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LEN)

tokenized_datasets = {}
for split in datasets:
    tokenized_datasets[split] = datasets[split].map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/16461 [00:00<?, ? examples/s]

Map:   0%|          | 0/3527 [00:00<?, ? examples/s]

Map:   0%|          | 0/3528 [00:00<?, ? examples/s]

Model Setup

In [ ]:
# Load Model
model_0= AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)
model_0.to(device)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    #return metric.compute(predictions=predictions, references=labels, average="macro")

# Training Arguments
args = TrainingArguments(
    output_dir="biobert-ade-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model_0,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    #tokenizer=tokenizer,
    data_collator=data_collator
)

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

Training

In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.153458,0.142654
2,0.076818,0.157062


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2058, training_loss=0.13715792860767004, metrics={'train_runtime': 470.5045, 'train_samples_per_second': 69.972, 'train_steps_per_second': 4.374, 'total_flos': 1186876498510440.0, 'train_loss': 0.13715792860767004, 'epoch': 2.0})

In [ ]:
print(f"Saving model to {SAVE_PATH}...")
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("Model and Tokenizer saved successfully.")

Saving model to ./saved_ade_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and Tokenizer saved successfully.


In [ ]:
def predict_ade_batch(texts, batch_size=16):
    model_0.eval() # Ensure model is in eval mode
    all_preds = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Classifying ADEs"):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            # Direct model inference
            outputs = model_0(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())

    return all_preds

In [ ]:
import pandas as pd

# Single user input
clinical_note = input("Enter clinical note: ").strip()

# Build dataframe identical to CSV pipeline
llm_df = pd.DataFrame({
    "clinical_note": [clinical_note]
})


Enter clinical note: Patient is a 58-year-old male with known hypertension and type 2 diabetes, currently on metformin 1000mg BID and amlodipine 5mg daily. He reports feeling well overall and his blood glucose has been trending in an acceptable range. Blood pressure today reads 138/88, slightly above target. Labs ordered for HbA1c and comprehensive metabolic panel.


In [ ]:
print("Running Step 1: ADE Classification...")

llm_df["has_ade"] = predict_ade_batch(llm_df["clinical_note"].tolist())

print("Prediction result:")
print(llm_df[["clinical_note", "has_ade"]])


Running Step 1: ADE Classification...


Classifying ADEs:   0%|          | 0/1 [00:00<?, ?it/s]

Prediction result:
                                       clinical_note  has_ade
0  Patient is a 58-year-old male with known hyper...        0


In [ ]:
print("Loading Step 2: Biomedical NER Model...")

ner_pipeline = pipeline(
    "token-classification",
    model="d4data/biomedical-ner-all",
    tokenizer="d4data/biomedical-ner-all",
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

Loading Step 2: Biomedical NER Model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
def extract_medical_entities(text):
    try:
        # Get raw predictions
        results = ner_pipeline(text)
    except Exception:
        return [], [],[], [], [], [], []

    # 1. Sort entities by their start position to ensure order
    results = sorted(results, key=lambda x: x['start'])

    merged_entities = []

    for entity in results:
        if not merged_entities:
            merged_entities.append(entity)
            continue

        prev = merged_entities[-1]

        # 2. CHECK ADJACENCY
        # If the current entity starts exactly where the previous one ended,
        # they are parts of the same word (e.g., "Di" + "zziness").
        if entity['start'] == prev['end']:
            # Extend the previous entity's endpoint to include this new part
            prev['end'] = entity['end']
            # We treat the label as the one from the start of the word
        else:
            merged_entities.append(entity)

    # 3. Extract final strings using the merged offsets
    drugs = set()
    diseases = set()
    sex=set()
    age=set()
    weight=set()
    dosage=set()
    frequency=set()



    for entity in merged_entities:
        start = entity['start']
        end = entity['end']
        label = entity['entity_group']

        # Slice the original text to get the clean, complete word
        full_word = text[start:end]

        if label in ['Medication','Chemical']:
            drugs.add(full_word)
        elif label in ['Disease_disorder', 'Sign_symptom']:
            diseases.add(full_word)
        elif label in ['Sex']:
            sex.add(full_word)
        elif label in ['Age']:
            age.add(full_word)
        elif label in ['Weight']:
            weight.add(full_word)
        elif label in ['Dosage']:
            dosage.add(full_word)
        elif label in ['Frequency']:
            frequency.add(full_word)

    return list(drugs), list(diseases),list(sex),list(age),list(weight),list(dosage),list(frequency)

In [ ]:
print("Running Step 2: Entity Extraction (Medical Entities)...")

tqdm.pandas()

cols = [
    'drugs',
    'diseases',
    'sex',
    'age',
    'weight',
    'dosage',
    'frequency'
]

llm_df[cols] = llm_df['clinical_note'].progress_apply(
    lambda x: pd.Series(extract_medical_entities(x))
)

llm_df.head()


Running Step 2: Entity Extraction (Medical Entities)...


  0%|          | 0/1 [00:00<?, ?it/s]

,clinical_note,has_ade,drugs,diseases,sex,age,weight,dosage,frequency
0,Patient is a 58-year-old male with known hyper...,0,"[1, amlodipine, metformin, H]",[feeling well],[male],[58-year-old],[],"[1000mg, 5mg]",[]


RELATIONAL EXTRACTION

In [ ]:
!pip install -q transformers[torch] datasets tqdm pandas scikit-learn

Creating Dataset(Negative pairings using ADE-neg.txt post NER, Positive pairings using Ade_corpus_v2_drug_ade_relation)

In [ ]:
import pandas as pd
import requests
from transformers import pipeline, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
import re

# Initialize
model_checkpoint_1 = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_1)
ner_pipe = pipeline("ner", model="d4data/biomedical-ner-all", aggregation_strategy="simple", device=0)

def wrap_entities(text, drug, ade):
    try:
        text = re.sub(f"({re.escape(drug)})", r"[DRUG]\1[/DRUG]", text, flags=re.IGNORECASE, count=1)
        text = re.sub(f"({re.escape(ade)})", r"[ADE]\1[/ADE]", text, flags=re.IGNORECASE, count=1)
    except: pass
    return text

# --- STEP A: Extract Negatives (Targeting ~2000-3000 samples) ---
url = "https://raw.githubusercontent.com/trunghlt/AdverseDrugReaction/refs/heads/master/ADE-Corpus-V2/ADE-NEG.txt"
lines = [l.split(' NEG ')[-1] for l in requests.get(url).text.strip().split('\n') if ' NEG ' in l]

neg_list = []
for text in tqdm(lines, desc="Scanning ADE-NEG"):
    ents = ner_pipe(text)
    # Using your confirmed labels: 'Chemical','Medication' and 'Disease_disorder','Sign_Symptom'
    drugs = [e['word'] for e in ents if e['entity_group'] in ['Chemical', 'Medication']]
    ades = [e['word'] for e in ents if e['entity_group'] in ['Disease_disorder', 'Sign_Symptom']]
    for d in drugs:
        for a in ades:
            neg_list.append({"text": wrap_entities(text, d, a), "label": 0})
    if len(neg_list) >= 5000: break # Safety cap

neg_df = pd.DataFrame(neg_list).drop_duplicates(subset=['text'])

# --- STEP B: Load and Downsample Positives ---
pos_raw = load_dataset("ade_corpus_v2", "Ade_corpus_v2_drug_ade_relation", split='train')
pos_list = []
for item in tqdm(pos_raw, desc="Processing Positives"):
    pos_list.append({"text": wrap_entities(item['text'], item['drug'], item['effect']), "label": 1})

pos_df = pd.DataFrame(pos_list).drop_duplicates(subset=['text'])

# --- STEP C: THE BALANCE ---
min_size = min(len(pos_df), len(neg_df))
balanced_df = pd.concat([
    pos_df.sample(min_size, random_state=42),
    neg_df.sample(min_size, random_state=42)
]).sample(frac=1).reset_index(drop=True)

print(f"\n✅ Dataset Balanced!")
print(f"Final Count: {len(balanced_df)} samples ({min_size} Pos, {min_size} Neg)")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Scanning ADE-NEG:  54%|█████▍    | 9003/16695 [01:28<01:15, 101.79it/s]


Ade_corpus_v2_drug_ade_relation/train-00(…):   0%|          | 0.00/491k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6821 [00:00<?, ? examples/s]

Processing Positives: 100%|██████████| 6821/6821 [00:01<00:00, 5681.47it/s]


✅ Dataset Balanced!
Final Count: 7128 samples (3564 Pos, 3564 Neg)


Tokeize and Train

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

# 1. Add markers to tokenizer
special_tokens = {'additional_special_tokens': ['[DRUG]', '[/DRUG]', '[ADE]', '[/ADE]']}
tokenizer.add_special_tokens(special_tokens)

# 2. Tokenize balanced data
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

hf_dataset = Dataset.from_pandas(balanced_df).map(tokenize_fn, batched=True)
train_test = hf_dataset.train_test_split(test_size=0.1)

# 3. Load Model and Resize
model_1 = AutoModelForSequenceClassification.from_pretrained(model_checkpoint_1, num_labels=2)
model_1.resize_token_embeddings(len(tokenizer))

# 4. Train
args = TrainingArguments(
    output_dir="./balanced_biobert_ade",
    max_steps=800,                # STOPS EXACTLY AT 800 STEPS
    logging_steps=10,             # Shows loss frequently
    eval_strategy="steps",
    eval_steps=100,               # Evaluate 8 times during training
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    weight_decay=0.01,
    load_best_model_at_end=True,  # Keeps the best version, not just the last
    report_to="none"
)

trainer = Trainer(model=model_1, args=args, train_dataset=train_test["train"], eval_dataset=train_test["test"])
trainer.train()

Map:   0%|          | 0/7128 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

Step,Training Loss,Validation Loss
100,0.057714,0.127769
200,0.107289,0.185373
300,0.123550,0.057824
400,0.161002,0.036157
500,0.003992,0.040020
600,0.027132,0.043703
700,0.000642,0.053408
800,0.000652,0.056317


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=800, training_loss=0.09744042665755842, metrics={'train_runtime': 353.2036, 'train_samples_per_second': 36.24, 'train_steps_per_second': 2.265, 'total_flos': 841889599388160.0, 'train_loss': 0.09744042665755842, 'epoch': 1.9950124688279303})

Test on CSV

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
import ast

# 1. Load your CSV file
# Ensure columns are named: 'clinical_note', 'drugs', 'conditions'
#path_to_csv = '/content/test.csv'
#df_user = pd.read_csv(path_to_csv)

model_1.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_1.to(device)

final_results = []

print(f"Analyzing {len(llm_df)} rows...")

# 2. Iterate and check every pair
for index, row in tqdm(llm_df.iterrows(), total=len(llm_df)):
    context = str(row.get('clinical_note', ''))

    # Explode the pipe-separated strings
    #drugs = [d.strip() for d in str(row.get('drugs', '')).split('|') if d.strip()]
    #diseases = [c.strip() for c in str(row.get('diseases', '')).split('|') if c.strip()]
    drugs = [
    d.strip()
    for d in ast.literal_eval(str(row.get("drugs", "[]")))
    if isinstance(d, str) and len(d.strip()) > 1
]
    diseases = [
    c.strip()
    for c in ast.literal_eval(str(row.get("diseases", "[]")))
    if isinstance(c, str) and len(c.strip()) > 1
]

    if not context or context == 'nan':
        continue

    for d in drugs:
        for c in diseases:
            # Wrap specific pair in markers [DRUG] and [ADE]
            # (Using the wrap_entities function defined in previous cells)
            marked_text = wrap_entities(context, d, c)

            # Predict
            inputs = tokenizer(marked_text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

            with torch.no_grad():
                logits = model_1(**inputs).logits
                prediction = torch.argmax(logits, dim=-1).item()
                confidence = torch.softmax(logits, dim=-1)[0][1].item()

            final_results.append({
                "original_row": index,
                "drug": d,
                "disease": c,
                "is_valid_ade": "Yes" if prediction == 1 else "No",
                "confidence": round(confidence, 4),
                "sentence_view": marked_text
            })

# 3. Save Final DataFrame
results_df = pd.DataFrame(final_results)
#results_df.to_csv("ade_pair_validation.csv", index=False)

print("\n--- Prediction Complete ---")
print(f"Total pairs checked: {len(results_df)}")
print(f"Validated ADEs found: {len(results_df[results_df['is_valid_ade'] == 'Yes'])}")

# Show the strongest 'Yes' predictions
display(results_df[results_df['is_valid_ade'] == "Yes"].sort_values(by="confidence", ascending=False).head(15))

Analyzing 1 rows...


100%|██████████| 1/1 [00:00<00:00, 36.34it/s]


--- Prediction Complete ---
Total pairs checked: 2
Validated ADEs found: 0


,original_row,drug,disease,is_valid_ade,confidence,sentence_view


Knowledge Base Integration

In [ ]:
import requests
import time

def fetch_reaction_meddrapt(drug, reaction, limit=50):
    base_url = "https://api.fda.gov/drug/event.json"

    search = (
        f'patient.drug.medicinalproduct:"{drug}" AND '
        f'patient.reaction.reactionmeddrapt:"{reaction}"'
    )

    params = {"search": search, "limit": limit}

    try:
        r = requests.get(base_url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
    except Exception:
        return []

    meddra_terms = set()

    for rec in data.get("results", []):
        for rxn in rec.get("patient", {}).get("reaction", []):
            pt = rxn.get("reactionmeddrapt")
            if pt:
                meddra_terms.add(pt.lower())

    return sorted(meddra_terms)


In [ ]:
results_df["openfda_reactionmeddrapt"] = [[] for _ in range(len(results_df))]


In [ ]:
print("Fetching OpenFDA MedDRA reaction terms...")

for idx, row in tqdm(
    results_df[results_df["is_valid_ade"] == "Yes"].iterrows(),
    total=len(results_df[results_df["is_valid_ade"] == "Yes"])
):
    drug = row["drug"].lower().strip()
    ade = row["disease"].lower().strip()

    meddra_pts = fetch_reaction_meddrapt(drug, ade)

    results_df.at[idx, "openfda_reactionmeddrapt"] = meddra_pts

    time.sleep(0.2)  # be polite


Fetching OpenFDA MedDRA reaction terms...


0it [00:00, ?it/s]


In [ ]:
display(results_df)


,original_row,drug,disease,is_valid_ade,confidence,sentence_view,openfda_reactionmeddrapt
0,0,amlodipine,feeling well,No,0.0016,Patient is a 58-year-old male with known hyper...,[]
1,0,metformin,feeling well,No,0.0012,Patient is a 58-year-old male with known hyper...,[]
